# PDF → Excel → SharePoint

Este notebook:
1. Lee todos los PDFs de una carpeta local
2. Usa Claude (IA) para extraer metadatos de cada PDF
3. Genera un Excel con columnas predefinidas
4. Sube cada fila como un elemento a una lista de SharePoint

**Columnas generadas:**
- `titulo` — nombre del archivo (sin extensión)
- `nombre` — título del documento extraído del contenido
- `pais` — país al que refiere el documento
- `emisor` — institución/organismo emisor
- `tipo_documento` — tipo de documento (ley, resolución, circular, etc.)
- `publicacion` — fecha de publicación del documento
- `tematica` — temática principal
- `descripcion` — resumen breve del contenido
- `palabras_claves` — palabras clave separadas por coma


## 1. Instalación de dependencias

In [ ]:
# Ejecutar sólo la primera vez
%pip install anthropic pdfplumber pandas openpyxl msal requests tqdm --quiet

## 2. Configuración — **editar aquí antes de ejecutar**

In [ ]:
import os

# ─── Claude ──────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "sk-ant-XXXXXX")  # ← tu API key de Anthropic

# ─── Carpeta con los PDFs ─────────────────────────────────────────────────────
PDF_FOLDER = "/ruta/a/tu/carpeta/pdfs"          # ← ruta absoluta a la carpeta con los PDFs

# ─── Excel de salida ──────────────────────────────────────────────────────────
OUTPUT_EXCEL = "documentos_regulatorios.xlsx"    # nombre del archivo Excel generado

# ─── SharePoint / Microsoft Graph API ────────────────────────────────────────
# Obtener en https://portal.azure.com → App registrations
SP_TENANT_ID    = "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"  # ← Tenant (Directory) ID
SP_CLIENT_ID    = "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"  # ← Application (Client) ID
SP_CLIENT_SECRET= "tu_client_secret_aqui"                 # ← Client Secret

# URL del sitio SharePoint (sin barra final)
# Ejemplo: https://miempresa.sharepoint.com/sites/MiSitio
SP_SITE_URL     = "https://miempresa.sharepoint.com/sites/MiSitio"

# Nombre EXACTO de la lista en SharePoint donde se subirán los registros
SP_LIST_NAME    = "Novedades Regulatorias"

# ─── Opciones avanzadas ───────────────────────────────────────────────────────
CLAUDE_MODEL    = "claude-opus-4-6"   # modelo de Claude a utilizar
MAX_PDF_CHARS   = 12_000              # máx. caracteres del PDF enviados a Claude
SKIP_ERRORS     = True                # True = omite PDFs con error y continúa

print("✅ Configuración cargada")

## 3. Funciones auxiliares

In [ ]:
import json
import re
import glob
import pdfplumber
import anthropic
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ─── 3.1 Extracción de texto desde PDF ───────────────────────────────────────
def extract_pdf_text(pdf_path: str, max_chars: int = MAX_PDF_CHARS) -> str:
    """Extrae el texto de un PDF (primeras `max_chars` caracteres)."""
    text_parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text_parts.append(page_text)
                    if sum(len(t) for t in text_parts) >= max_chars:
                        break
        full_text = "\n".join(text_parts)
        return full_text[:max_chars]
    except Exception as e:
        raise RuntimeError(f"Error leyendo {pdf_path}: {e}")


# ─── 3.2 Extracción de metadatos con Claude ───────────────────────────────────
EXTRACTION_PROMPT = """\
Eres un experto en análisis de documentos regulatorios. \
A partir del texto del documento que se proporciona a continuación, extrae la siguiente información \
y devuélvela ÚNICAMENTE como un objeto JSON válido, sin texto adicional antes ni después.

Campos a extraer:
- "nombre": título oficial del documento (string)
- "pais": país al que pertenece o al que refiere el documento (string, en español)
- "emisor": institución u organismo que emite el documento (string)
- "tipo_documento": tipo de documento, ej: ley, decreto, resolución, circular, norma, reglamento, guía, informe, etc. (string)
- "publicacion": fecha de publicación o vigencia en formato DD/MM/AAAA; si no se encuentra exactamente, usa el año (string)
- "tematica": temática o área principal del documento, ej: regulación bancaria, protección al consumidor, \
mercado de valores, ciberseguridad, etc. (string)
- "descripcion": resumen conciso del contenido y propósito del documento, máximo 3 oraciones (string)
- "palabras_claves": lista de 5-8 palabras clave separadas por coma (string)

Si algún campo no se puede determinar con certeza, usa null.

Texto del documento:
---------------------
{text}
---------------------

Responde SÓLO con el JSON:"""


_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def extract_metadata_with_claude(text: str, filename: str) -> dict:
    """Llama a Claude para extraer metadatos del texto del PDF."""
    prompt = EXTRACTION_PROMPT.format(text=text)
    response = _client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.content[0].text.strip()
    # Eliminar bloques markdown si Claude los añade
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
    raw = re.sub(r"```\s*$", "", raw, flags=re.MULTILINE).strip()
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Claude no devolvió JSON válido para '{filename}': {e}\nRespuesta: {raw[:300]}")
    return data


print("✅ Funciones cargadas")

## 4. Procesar PDFs y generar el Excel

In [ ]:
pdf_files = sorted(glob.glob(os.path.join(PDF_FOLDER, "**/*.pdf"), recursive=True))
pdf_files += sorted(glob.glob(os.path.join(PDF_FOLDER, "*.pdf")))
pdf_files = list(dict.fromkeys(pdf_files))  # deduplicar

print(f"📁 {len(pdf_files)} PDFs encontrados en: {PDF_FOLDER}")
if not pdf_files:
    raise FileNotFoundError(f"No se encontraron PDFs en: {PDF_FOLDER}")

rows = []
errores = []

for pdf_path in tqdm(pdf_files, desc="Procesando PDFs"):
    filename = Path(pdf_path).stem  # nombre sin extensión
    try:
        # Extraer texto
        text = extract_pdf_text(pdf_path)
        if not text.strip():
            raise ValueError("PDF sin texto extraíble (puede ser imagen escaneada)")

        # Extraer metadatos con Claude
        meta = extract_metadata_with_claude(text, filename)

        rows.append({
            "titulo":          filename,
            "nombre":          meta.get("nombre"),
            "pais":            meta.get("pais"),
            "emisor":          meta.get("emisor"),
            "tipo_documento":  meta.get("tipo_documento"),
            "publicacion":     meta.get("publicacion"),
            "tematica":        meta.get("tematica"),
            "descripcion":     meta.get("descripcion"),
            "palabras_claves": meta.get("palabras_claves"),
        })
    except Exception as e:
        msg = f"❌ Error en '{filename}': {e}"
        errores.append({"archivo": filename, "error": str(e)})
        if SKIP_ERRORS:
            print(msg)
        else:
            raise

df = pd.DataFrame(rows)
df.to_excel(OUTPUT_EXCEL, index=False, engine="openpyxl")

print(f"\n✅ Excel generado: {OUTPUT_EXCEL}")
print(f"   Filas procesadas: {len(rows)}")
print(f"   Errores:          {len(errores)}")
if errores:
    print("\nArchivos con error:")
    for e in errores:
        print(f"  • {e['archivo']}: {e['error']}")
df.head()

## 5. Subir a la lista de SharePoint

### Prerrequisitos en Azure / SharePoint

1. Ir a [portal.azure.com](https://portal.azure.com) → **Azure Active Directory** → **App registrations** → **New registration**
2. Anotar el **Tenant ID** y el **Application (Client) ID**
3. En *Certificates & secrets* → crear un **Client secret**
4. En *API permissions* → agregar:
   - `Sites.ReadWrite.All` (Microsoft Graph, tipo **Application**)
   - Hacer clic en **Grant admin consent**
5. En la lista de SharePoint, asegurarse de que los **nombres internos de columna** coincidan con los de abajo (o ajustar `COLUMN_MAPPING`).

In [ ]:
import msal
import requests
from urllib.parse import urlparse

# ─── Mapeo: columna del DataFrame → nombre interno de columna en SharePoint ───
# Cambiar los valores si los nombres internos de tus columnas son distintos.
# El nombre interno se ve en SharePoint: Configuración de lista → columna → URL (?Field=...)
COLUMN_MAPPING = {
    "titulo":          "Titulo",
    "nombre":          "Nombre",
    "pais":            "Pais",
    "emisor":          "Emisor",
    "tipo_documento":  "TipoDocumento",
    "publicacion":     "Publicacion",
    "tematica":        "Tematica",
    "descripcion":     "Descripcion",
    "palabras_claves": "PalabrasClave",
}

print("✅ Mapeo de columnas configurado")

In [ ]:
# ─── 5.1 Autenticación con MSAL (Client Credentials) ─────────────────────────

def get_access_token() -> str:
    """Obtiene un access token de Microsoft Graph usando Client Credentials."""
    authority = f"https://login.microsoftonline.com/{SP_TENANT_ID}"
    app = msal.ConfidentialClientApplication(
        client_id=SP_CLIENT_ID,
        client_credential=SP_CLIENT_SECRET,
        authority=authority,
    )
    result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"Error obteniendo token: {result.get('error_description', result)}")
    return result["access_token"]


# ─── 5.2 Helpers de Graph API ─────────────────────────────────────────────────

GRAPH_BASE = "https://graph.microsoft.com/v1.0"

def graph_get(token: str, url: str) -> dict:
    resp = requests.get(url, headers={"Authorization": f"Bearer {token}"})
    resp.raise_for_status()
    return resp.json()


def graph_post(token: str, url: str, body: dict) -> dict:
    resp = requests.post(
        url,
        json=body,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    )
    resp.raise_for_status()
    return resp.json()


def get_site_id(token: str, site_url: str) -> str:
    """Obtiene el siteId de Graph a partir de la URL del sitio SharePoint."""
    parsed = urlparse(site_url)
    hostname = parsed.netloc                     # ej. miempresa.sharepoint.com
    site_path = parsed.path.lstrip("/")          # ej. sites/MiSitio
    url = f"{GRAPH_BASE}/sites/{hostname}:/{site_path}"
    data = graph_get(token, url)
    return data["id"]


def get_list_id(token: str, site_id: str, list_name: str) -> str:
    """Obtiene el listId de una lista por su nombre."""
    url = f"{GRAPH_BASE}/sites/{site_id}/lists"
    data = graph_get(token, url)
    for lst in data.get("value", []):
        if lst["displayName"].strip().lower() == list_name.strip().lower():
            return lst["id"]
    available = [l["displayName"] for l in data.get("value", [])]
    raise ValueError(
        f"Lista '{list_name}' no encontrada.\nListas disponibles: {available}"
    )


def create_list_item(token: str, site_id: str, list_id: str, fields: dict) -> dict:
    """Crea un elemento en una lista de SharePoint."""
    url = f"{GRAPH_BASE}/sites/{site_id}/lists/{list_id}/items"
    body = {"fields": fields}
    return graph_post(token, url, body)


print("✅ Funciones de SharePoint cargadas")

In [ ]:
# ─── 5.3 Conectar a SharePoint y obtener IDs ─────────────────────────────────

print("🔐 Autenticando con Microsoft Graph...")
token = get_access_token()
print("✅ Token obtenido")

print(f"🔍 Buscando sitio: {SP_SITE_URL}")
site_id = get_site_id(token, SP_SITE_URL)
print(f"✅ Site ID: {site_id}")

print(f"🔍 Buscando lista: {SP_LIST_NAME}")
list_id = get_list_id(token, site_id, SP_LIST_NAME)
print(f"✅ List ID: {list_id}")

In [ ]:
# ─── 5.4 Subir filas del Excel a la lista de SharePoint ──────────────────────

# Leer el Excel generado (puedes saltar si el DataFrame `df` ya está en memoria)
df_upload = pd.read_excel(OUTPUT_EXCEL, engine="openpyxl")
df_upload = df_upload.where(pd.notna(df_upload), None)  # reemplaza NaN por None

success = 0
failed  = []

for idx, row in tqdm(df_upload.iterrows(), total=len(df_upload), desc="Subiendo a SharePoint"):
    # Construir dict de campos usando el mapeo de columnas
    fields = {}
    for df_col, sp_col in COLUMN_MAPPING.items():
        value = row.get(df_col)
        if value is not None:
            fields[sp_col] = str(value)

    titulo = row.get("titulo", f"fila_{idx}")
    try:
        # Renovar token cada 50 elementos (expira en 1 h)
        if idx % 50 == 0 and idx > 0:
            token = get_access_token()
        create_list_item(token, site_id, list_id, fields)
        success += 1
    except requests.HTTPError as e:
        failed.append({"titulo": titulo, "error": str(e), "detalle": e.response.text[:300]})
        if not SKIP_ERRORS:
            raise

print(f"\n✅ Subidos correctamente: {success}/{len(df_upload)}")
if failed:
    print(f"❌ Errores: {len(failed)}")
    for f in failed:
        print(f"  • {f['titulo']}: {f['error']}")
        print(f"    Detalle: {f['detalle']}")

## 6. (Opcional) Ver resumen del Excel

In [ ]:
import pandas as pd
df_check = pd.read_excel(OUTPUT_EXCEL, engine="openpyxl")
print(f"Total filas: {len(df_check)}")
display(df_check)

---
## Notas de configuración

### Permisos necesarios en Azure App Registration
| Permiso | Tipo | Para qué |
|---|---|---|
| `Sites.ReadWrite.All` | Application | Leer/escribir listas de SharePoint |

> Después de agregar los permisos hay que hacer clic en **"Grant admin consent"**.

### Nombres internos de columnas en SharePoint
Para ver el nombre interno de una columna en SharePoint:
1. Ir a la lista → **Configuración de lista** → hacer clic en el nombre de la columna
2. En la URL de la página de configuración, buscar el parámetro `?Field=NombreInterno`
3. Actualizar `COLUMN_MAPPING` en la celda 5 con esos nombres internos

### PDFs escaneados (sólo imágenes)
Si un PDF no tiene texto extraíble (sólo contiene imágenes escaneadas), el proceso mostrará un error.
Para esos casos se puede usar OCR con `pytesseract` o el servicio Azure Document Intelligence.
